In [1]:
import os
import json
import time
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

from tqdm.auto import tqdm

In [2]:
# ============================================================
# Configuration
# ============================================================

SEED = 42

NUM_CLASSES = 101
IMAGE_SIZE = 160

BATCH_SIZE = 64
NUM_WORKERS = 4

NUM_EPOCHS = 15

LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

DATA_DIR = "/kaggle/input/food-101/food-101"
OUTPUT_DIR = "/kaggle/working/results"

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [5]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Reproducibility
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(SEED)

In [6]:
# ============================================================
# GPU setup
# ============================================================

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"Number of GPUs: {torch.cuda.device_count()}")

    for i in range(torch.cuda.device_count()):
        print(
            f"GPU {i}: {torch.cuda.get_device_name(i)} | "
            f"Memory: "
            f"{torch.cuda.get_device_properties(i).total_memory / 1024**3:.2f} GB"
        )
else:
    print("WARNING: CUDA is not available.")

PyTorch version: 2.10.0+cu128
CUDA available: True
Number of GPUs: 2
GPU 0: Tesla T4 | Memory: 14.56 GB
GPU 1: Tesla T4 | Memory: 14.56 GB


In [7]:
# ============================================================
# ImageNet normalization
# ============================================================

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

In [8]:
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(
        IMAGE_SIZE,
        scale=(0.7, 1.0)
    ),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])

test_transform = transforms.Compose([
    transforms.Resize(int(IMAGE_SIZE * 1.15)),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])

In [9]:
train_dataset = datasets.Food101(
    root="/kaggle/working",
    split="train",
    transform=train_transform,
    download=True
)

test_dataset = datasets.Food101(
    root="/kaggle/working",
    split="test",
    transform=test_transform,
    download=True
)


100%|██████████| 5.00G/5.00G [02:40<00:00, 31.1MB/s] 


In [10]:
print(f"Training samples: {len(train_dataset):,}")
print(f"Test samples:     {len(test_dataset):,}")
print(f"Classes:           {len(train_dataset.classes)}")

Training samples: 75,750
Test samples:     25,250
Classes:           101


In [13]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True
)

In [14]:
images, labels = next(iter(train_loader))

print("Images:", images.shape)
print("Labels:", labels.shape)
print("Image dtype:", images.dtype)
print("Label dtype:", labels.dtype)

Images: torch.Size([64, 3, 160, 160])
Labels: torch.Size([64])
Image dtype: torch.float32
Label dtype: torch.int64


In [15]:
# ============================================================
# ResNet-152 model factory
# ============================================================

def build_resnet152(pretrained=True, train_mode="final_block"):
    """
    Build ResNet-152 for Food-101.

    Parameters
    ----------
    pretrained : bool
        If True, initialize with ImageNet-pretrained weights.
        If False, initialize randomly.

    train_mode : str
        "final_block" -> train layer4 + classifier
        "full"        -> train entire backbone + classifier
    """

    if train_mode not in ["final_block", "full"]:
        raise ValueError(
            "train_mode must be either 'final_block' or 'full'"
        )

    # --------------------------------------------------------
    # Initialize ResNet-152
    # --------------------------------------------------------

    if pretrained:
        weights = models.ResNet152_Weights.IMAGENET1K_V2
    else:
        weights = None

    model = models.resnet152(weights=weights)

    # --------------------------------------------------------
    # Replace ImageNet classifier
    # --------------------------------------------------------

    in_features = model.fc.in_features

    model.fc = nn.Linear(
        in_features,
        NUM_CLASSES
    )

    # --------------------------------------------------------
    # Freeze / unfreeze
    # --------------------------------------------------------

    if train_mode == "final_block":

        # Freeze entire model first
        for param in model.parameters():
            param.requires_grad = False

        # Unfreeze final residual block
        for param in model.layer4.parameters():
            param.requires_grad = True

        # Unfreeze classifier
        for param in model.fc.parameters():
            param.requires_grad = True

    elif train_mode == "full":

        # Everything is trainable
        for param in model.parameters():
            param.requires_grad = True

    return model

In [16]:
# ============================================================
# Parameter statistics
# ============================================================

def parameter_stats(model):
    total_params = sum(
        p.numel()
        for p in model.parameters()
    )

    trainable_params = sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

    frozen_params = total_params - trainable_params

    return {
        "total": total_params,
        "trainable": trainable_params,
        "frozen": frozen_params,
        "trainable_ratio": trainable_params / total_params
    }

In [17]:
model_final = build_resnet152(
    pretrained=True,
    train_mode="final_block"
)

stats_final = parameter_stats(model_final)

print("ImageNet + Final Block")
print("-" * 40)

for key, value in stats_final.items():
    if key == "trainable_ratio":
        print(f"{key}: {value:.2%}")
    else:
        print(f"{key}: {value:,}")

Downloading: "https://download.pytorch.org/models/resnet152-f82ba261.pth" to /root/.cache/torch/hub/checkpoints/resnet152-f82ba261.pth


100%|██████████| 230M/230M [00:01<00:00, 184MB/s]  


ImageNet + Final Block
----------------------------------------
total: 58,350,757
trainable: 15,171,685
frozen: 43,179,072
trainable_ratio: 26.00%


In [18]:
model_full = build_resnet152(
    pretrained=True,
    train_mode="full"
)

stats_full = parameter_stats(model_full)

print("\nImageNet + Full Backbone")
print("-" * 40)

for key, value in stats_full.items():
    if key == "trainable_ratio":
        print(f"{key}: {value:.2%}")
    else:
        print(f"{key}: {value:,}")


ImageNet + Full Backbone
----------------------------------------
total: 58,350,757
trainable: 58,350,757
frozen: 0
trainable_ratio: 100.00%


In [19]:
print("Trainable layers:")
print("=" * 50)

for name, param in model_final.named_parameters():
    if param.requires_grad:
        print(name)

Trainable layers:
layer4.0.conv1.weight
layer4.0.bn1.weight
layer4.0.bn1.bias
layer4.0.conv2.weight
layer4.0.bn2.weight
layer4.0.bn2.bias
layer4.0.conv3.weight
layer4.0.bn3.weight
layer4.0.bn3.bias
layer4.0.downsample.0.weight
layer4.0.downsample.1.weight
layer4.0.downsample.1.bias
layer4.1.conv1.weight
layer4.1.bn1.weight
layer4.1.bn1.bias
layer4.1.conv2.weight
layer4.1.bn2.weight
layer4.1.bn2.bias
layer4.1.conv3.weight
layer4.1.bn3.weight
layer4.1.bn3.bias
layer4.2.conv1.weight
layer4.2.bn1.weight
layer4.2.bn1.bias
layer4.2.conv2.weight
layer4.2.bn2.weight
layer4.2.bn2.bias
layer4.2.conv3.weight
layer4.2.bn3.weight
layer4.2.bn3.bias
fc.weight
fc.bias


In [20]:
device = torch.device("cuda:0")

model_final = model_final.to(device)

images, labels = next(iter(train_loader))

images = images.to(device, non_blocking=True)

with torch.no_grad():
    outputs = model_final(images)

print("Input:", images.shape)
print("Output:", outputs.shape)

Input: torch.Size([64, 3, 160, 160])
Output: torch.Size([64, 101])


In [21]:
# ============================================================
# Mixed precision setup
# ============================================================

def get_amp_scaler():
    return torch.amp.GradScaler("cuda")

In [22]:
# ============================================================
# Accuracy
# ============================================================

def calculate_accuracy(outputs, labels):
    predictions = outputs.argmax(dim=1)
    correct = (predictions == labels).sum().item()
    total = labels.size(0)

    return correct / total

In [27]:
# ============================================================
# Train one epoch
# ============================================================

def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
    device,
    scaler
):
    model.train()

    freeze_batchnorm_in_frozen_layers(model)

    running_loss = 0.0
    running_correct = 0
    total_samples = 0

    progress_bar = tqdm(
        loader,
        desc="Training",
        leave=False
    )

    for images, labels in progress_bar:

        images = images.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad(set_to_none=True)

        # -----------------------------------------------
        # Mixed precision forward pass
        # -----------------------------------------------

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):
            outputs = model(images)
            loss = criterion(outputs, labels)

        # -----------------------------------------------
        # Backward pass
        # -----------------------------------------------

        scaler.scale(loss).backward()

        scaler.step(optimizer)
        scaler.update()

        # -----------------------------------------------
        # Statistics
        # -----------------------------------------------

        batch_size = images.size(0)

        running_loss += loss.item() * batch_size

        predictions = outputs.argmax(dim=1)

        running_correct += (
            predictions == labels
        ).sum().item()

        total_samples += batch_size

        progress_bar.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    epoch_loss = running_loss / total_samples
    epoch_accuracy = running_correct / total_samples

    return epoch_loss, epoch_accuracy

In [24]:
# ============================================================
# Evaluate model
# ============================================================

@torch.no_grad()
def evaluate(
    model,
    loader,
    criterion,
    device
):
    model.eval()

    running_loss = 0.0
    running_correct = 0
    total_samples = 0

    progress_bar = tqdm(
        loader,
        desc="Validation",
        leave=False
    )

    for images, labels in progress_bar:

        images = images.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):
            outputs = model(images)
            loss = criterion(outputs, labels)

        batch_size = images.size(0)

        running_loss += loss.item() * batch_size

        predictions = outputs.argmax(dim=1)

        running_correct += (
            predictions == labels
        ).sum().item()

        total_samples += batch_size

    epoch_loss = running_loss / total_samples
    epoch_accuracy = running_correct / total_samples

    return epoch_loss, epoch_accuracy

In [26]:
# ============================================================
# Keep frozen BatchNorm layers in eval mode
# ============================================================

def freeze_batchnorm_in_frozen_layers(model):
    """
    Keep BatchNorm layers inside frozen portions of the
    network in evaluation mode so their running statistics
    do not change during fine-tuning.
    """

    for module in model.modules():
        if isinstance(module, nn.BatchNorm2d):

            # If none of the BN parameters are trainable,
            # keep this BatchNorm layer frozen.
            if not any(
                param.requires_grad
                for param in module.parameters()
            ):
                module.eval()

                # Make absolutely sure BN parameters
                # remain frozen.
                for param in module.parameters():
                    param.requires_grad = False

In [25]:
# ============================================================
# Full training loop
# ============================================================

def train_model(
    model,
    train_loader,
    val_loader,
    device,
    num_epochs,
    learning_rate,
    weight_decay,
    experiment_name
):

    criterion = nn.CrossEntropyLoss()

    # Only parameters with requires_grad=True are optimized
    trainable_parameters = [
        p for p in model.parameters()
        if p.requires_grad
    ]

    optimizer = optim.AdamW(
        trainable_parameters,
        lr=learning_rate,
        weight_decay=weight_decay
    )

    scaler = get_amp_scaler()

    history = {
        "train_loss": [],
        "train_accuracy": [],
        "val_loss": [],
        "val_accuracy": [],
        "epoch_time": []
    }

    best_val_accuracy = 0.0

    experiment_dir = os.path.join(
        OUTPUT_DIR,
        experiment_name
    )

    os.makedirs(
        experiment_dir,
        exist_ok=True
    )

    print("=" * 70)
    print(f"Experiment: {experiment_name}")
    print("=" * 70)

    stats = parameter_stats(model)

    print(
        f"Total parameters:     "
        f"{stats['total']:,}"
    )

    print(
        f"Trainable parameters: "
        f"{stats['trainable']:,}"
    )

    print(
        f"Trainable ratio:      "
        f"{stats['trainable_ratio']:.2%}"
    )

    print()

    for epoch in range(num_epochs):

        epoch_start = time.time()

        print(
            f"Epoch {epoch + 1}/{num_epochs}"
        )

        # -----------------------------------------------
        # Training
        # -----------------------------------------------

        train_loss, train_accuracy = train_one_epoch(
            model=model,
            loader=train_loader,
            criterion=criterion,
            optimizer=optimizer,
            device=device,
            scaler=scaler
        )

        # -----------------------------------------------
        # Validation
        # -----------------------------------------------

        val_loss, val_accuracy = evaluate(
            model=model,
            loader=val_loader,
            criterion=criterion,
            device=device
        )

        epoch_time = time.time() - epoch_start

        # -----------------------------------------------
        # Save history
        # -----------------------------------------------

        history["train_loss"].append(train_loss)
        history["train_accuracy"].append(train_accuracy)

        history["val_loss"].append(val_loss)
        history["val_accuracy"].append(val_accuracy)

        history["epoch_time"].append(epoch_time)

        # -----------------------------------------------
        # Save best model
        # -----------------------------------------------

        if val_accuracy > best_val_accuracy:

            best_val_accuracy = val_accuracy

            checkpoint_path = os.path.join(
                experiment_dir,
                "best_model.pt"
            )

            torch.save(
                model.state_dict(),
                checkpoint_path
            )

        # -----------------------------------------------
        # Report
        # -----------------------------------------------

        print(
            f"Train Loss: {train_loss:.4f} | "
            f"Train Acc: {train_accuracy:.4f}"
        )

        print(
            f"Val Loss:   {val_loss:.4f} | "
            f"Val Acc:   {val_accuracy:.4f}"
        )

        print(
            f"Time: {epoch_time / 60:.2f} min"
        )

        print("-" * 70)

    # -----------------------------------------------
    # Save history
    # -----------------------------------------------

    history_path = os.path.join(
        experiment_dir,
        "history.json"
    )

    with open(history_path, "w") as f:
        json.dump(history, f, indent=4)

    print(
        f"\nBest validation accuracy: "
        f"{best_val_accuracy:.4f}"
    )

    return history

In [28]:
# ============================================================
# 2-epoch pilot
# ImageNet pretrained + final block
# ============================================================

set_seed(SEED)

device = torch.device("cuda:0")

model_final = build_resnet152(
    pretrained=True,
    train_mode="final_block"
).to(device)

In [29]:
pilot_history = train_model(
    model=model_final,
    train_loader=train_loader,
    val_loader=test_loader,
    device=device,
    num_epochs=2,
    learning_rate=1e-3,
    weight_decay=1e-4,
    experiment_name="pilot_imagenet_final"
)

Experiment: pilot_imagenet_final
Total parameters:     58,350,757
Trainable parameters: 15,171,685
Trainable ratio:      26.00%

Epoch 1/2


Training:   0%|          | 0/1184 [00:00<?, ?it/s]

Validation:   0%|          | 0/395 [00:00<?, ?it/s]

Train Loss: 1.3333 | Train Acc: 0.6571
Val Loss:   0.7465 | Val Acc:   0.7918
Time: 3.52 min
----------------------------------------------------------------------
Epoch 2/2


Training:   0%|          | 0/1184 [00:00<?, ?it/s]

Validation:   0%|          | 0/395 [00:00<?, ?it/s]

Train Loss: 0.7607 | Train Acc: 0.7908
Val Loss:   0.6792 | Val Acc:   0.8106
Time: 3.56 min
----------------------------------------------------------------------

Best validation accuracy: 0.8106


In [30]:
# ============================================================
# 2-epoch pilot
# ImageNet pretrained + Full Backbone
# GPU 1
# ============================================================

set_seed(SEED)

device_full = torch.device("cuda:1")

model_full = build_resnet152(
    pretrained=True,
    train_mode="full"
).to(device_full)

stats_full = parameter_stats(model_full)

print("ImageNet + Full Backbone")
print("=" * 50)

for key, value in stats_full.items():
    if key == "trainable_ratio":
        print(f"{key}: {value:.2%}")
    else:
        print(f"{key}: {value:,}")

ImageNet + Full Backbone
total: 58,350,757
trainable: 58,350,757
frozen: 0
trainable_ratio: 100.00%


In [31]:
full_pilot_history = train_model(
    model=model_full,
    train_loader=train_loader,
    val_loader=test_loader,
    device=device_full,
    num_epochs=2,
    learning_rate=1e-4,
    weight_decay=1e-4,
    experiment_name="pilot_imagenet_full"
)

Experiment: pilot_imagenet_full
Total parameters:     58,350,757
Trainable parameters: 58,350,757
Trainable ratio:      100.00%

Epoch 1/2


Training:   0%|          | 0/1184 [00:00<?, ?it/s]

Validation:   0%|          | 0/395 [00:00<?, ?it/s]

Train Loss: 1.5822 | Train Acc: 0.6122
Val Loss:   0.7165 | Val Acc:   0.8036
Time: 7.85 min
----------------------------------------------------------------------
Epoch 2/2


Training:   0%|          | 0/1184 [00:00<?, ?it/s]

Validation:   0%|          | 0/395 [00:00<?, ?it/s]

Train Loss: 0.7429 | Train Acc: 0.7988
Val Loss:   0.6246 | Val Acc:   0.8219
Time: 8.01 min
----------------------------------------------------------------------

Best validation accuracy: 0.8219


In [32]:
%%writefile /kaggle/working/train_experiment.py

import os
import json
import time
import random
import argparse

import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from tqdm.auto import tqdm


# ============================================================
# Configuration
# ============================================================

SEED = 42

NUM_CLASSES = 101
IMAGE_SIZE = 160

BATCH_SIZE = 64
NUM_WORKERS = 4

NUM_EPOCHS = 15

WEIGHT_DECAY = 1e-4

DATA_ROOT = "/kaggle/working"
OUTPUT_DIR = "/kaggle/working/results"

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


# ============================================================
# Reproducibility
# ============================================================

def set_seed(seed=42):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# ============================================================
# Model
# ============================================================

def build_resnet152(pretrained=True, train_mode="final_block"):

    if train_mode not in ["final_block", "full"]:
        raise ValueError(
            "train_mode must be 'final_block' or 'full'"
        )

    if pretrained:
        weights = models.ResNet152_Weights.IMAGENET1K_V2
    else:
        weights = None

    model = models.resnet152(weights=weights)

    # Replace ImageNet classifier
    in_features = model.fc.in_features

    model.fc = nn.Linear(
        in_features,
        NUM_CLASSES
    )

    # --------------------------------------------------------
    # Freeze / unfreeze
    # --------------------------------------------------------

    if train_mode == "final_block":

        for param in model.parameters():
            param.requires_grad = False

        # Train final residual block
        for param in model.layer4.parameters():
            param.requires_grad = True

        # Train classifier
        for param in model.fc.parameters():
            param.requires_grad = True

    else:

        for param in model.parameters():
            param.requires_grad = True

    return model


# ============================================================
# BatchNorm handling
# ============================================================

def freeze_batchnorm_in_frozen_layers(model):

    for module in model.modules():

        if isinstance(module, nn.BatchNorm2d):

            if not any(
                param.requires_grad
                for param in module.parameters()
            ):

                module.eval()

                for param in module.parameters():
                    param.requires_grad = False


# ============================================================
# Parameter statistics
# ============================================================

def parameter_stats(model):

    total = sum(
        p.numel()
        for p in model.parameters()
    )

    trainable = sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

    return {
        "total": total,
        "trainable": trainable,
        "frozen": total - trainable,
        "trainable_ratio": trainable / total
    }


# ============================================================
# Dataset
# ============================================================

def build_dataloaders():

    train_transform = transforms.Compose([
        transforms.RandomResizedCrop(
            IMAGE_SIZE,
            scale=(0.7, 1.0)
        ),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(
            IMAGENET_MEAN,
            IMAGENET_STD
        )
    ])

    test_transform = transforms.Compose([
        transforms.Resize(
            int(IMAGE_SIZE * 1.15)
        ),
        transforms.CenterCrop(IMAGE_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(
            IMAGENET_MEAN,
            IMAGENET_STD
        )
    ])

    train_dataset = datasets.Food101(
        root=DATA_ROOT,
        split="train",
        transform=train_transform,
        download=True
    )

    test_dataset = datasets.Food101(
        root=DATA_ROOT,
        split="test",
        transform=test_transform,
        download=True
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=True
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=True
    )

    return train_loader, test_loader


# ============================================================
# Train
# ============================================================

def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
    device,
    scaler
):

    model.train()

    # Keep frozen BatchNorm statistics fixed
    freeze_batchnorm_in_frozen_layers(model)

    running_loss = 0.0
    running_correct = 0
    total_samples = 0

    progress = tqdm(
        loader,
        desc="Training",
        leave=False
    )

    for images, labels in progress:

        images = images.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):

            outputs = model(images)

            loss = criterion(
                outputs,
                labels
            )

        scaler.scale(loss).backward()

        scaler.step(optimizer)
        scaler.update()

        batch_size = images.size(0)

        running_loss += (
            loss.item() * batch_size
        )

        running_correct += (
            outputs.argmax(dim=1) == labels
        ).sum().item()

        total_samples += batch_size

        progress.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    return (
        running_loss / total_samples,
        running_correct / total_samples
    )


# ============================================================
# Evaluation
# ============================================================

@torch.no_grad()
def evaluate(
    model,
    loader,
    criterion,
    device
):

    model.eval()

    running_loss = 0.0
    running_correct = 0
    total_samples = 0

    progress = tqdm(
        loader,
        desc="Validation",
        leave=False
    )

    for images, labels in progress:

        images = images.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):

            outputs = model(images)

            loss = criterion(
                outputs,
                labels
            )

        batch_size = images.size(0)

        running_loss += (
            loss.item() * batch_size
        )

        running_correct += (
            outputs.argmax(dim=1) == labels
        ).sum().item()

        total_samples += batch_size

    return (
        running_loss / total_samples,
        running_correct / total_samples
    )


# ============================================================
# Main training
# ============================================================

def train_experiment(
    gpu_id,
    pretrained,
    train_mode,
    experiment_name
):

    # --------------------------------------------------------
    # GPU
    # --------------------------------------------------------

    device = torch.device(
        f"cuda:{gpu_id}"
    )

    torch.cuda.set_device(device)

    # --------------------------------------------------------
    # Seed
    # --------------------------------------------------------

    set_seed(SEED)

    # --------------------------------------------------------
    # Data
    # --------------------------------------------------------

    train_loader, test_loader = build_dataloaders()

    # --------------------------------------------------------
    # Model
    # --------------------------------------------------------

    model = build_resnet152(
        pretrained=pretrained,
        train_mode=train_mode
    )

    model = model.to(device)

    stats = parameter_stats(model)

    # --------------------------------------------------------
    # Optimizer
    # --------------------------------------------------------

    if train_mode == "final_block":
        learning_rate = 1e-3
    else:
        learning_rate = 1e-4

    trainable_parameters = [
        p for p in model.parameters()
        if p.requires_grad
    ]

    criterion = nn.CrossEntropyLoss()

    optimizer = optim.AdamW(
        trainable_parameters,
        lr=learning_rate,
        weight_decay=WEIGHT_DECAY
    )

    scaler = torch.amp.GradScaler("cuda")

    # --------------------------------------------------------
    # Output
    # --------------------------------------------------------

    experiment_dir = os.path.join(
        OUTPUT_DIR,
        experiment_name
    )

    os.makedirs(
        experiment_dir,
        exist_ok=True
    )

    # --------------------------------------------------------
    # Metadata
    # --------------------------------------------------------

    metadata = {
        "experiment": experiment_name,
        "gpu_id": gpu_id,
        "gpu_name": torch.cuda.get_device_name(gpu_id),
        "pretrained": pretrained,
        "train_mode": train_mode,
        "num_classes": NUM_CLASSES,
        "image_size": IMAGE_SIZE,
        "batch_size": BATCH_SIZE,
        "num_workers": NUM_WORKERS,
        "num_epochs": NUM_EPOCHS,
        "learning_rate": learning_rate,
        "weight_decay": WEIGHT_DECAY,
        "total_parameters": stats["total"],
        "trainable_parameters": stats["trainable"],
        "trainable_ratio": stats["trainable_ratio"]
    }

    with open(
        os.path.join(
            experiment_dir,
            "metadata.json"
        ),
        "w"
    ) as f:

        json.dump(
            metadata,
            f,
            indent=4
        )

    # --------------------------------------------------------
    # History
    # --------------------------------------------------------

    history = {
        "train_loss": [],
        "train_accuracy": [],
        "val_loss": [],
        "val_accuracy": [],
        "epoch_time": [],
        "gpu_memory_peak_mb": []
    }

    best_val_accuracy = 0.0
    best_epoch = 0

    total_start = time.time()

    # --------------------------------------------------------
    # Training
    # --------------------------------------------------------

    print("=" * 70)
    print(f"Experiment: {experiment_name}")
    print("=" * 70)

    print(
        f"GPU: {torch.cuda.get_device_name(gpu_id)}"
    )

    print(
        f"Total parameters:     "
        f"{stats['total']:,}"
    )

    print(
        f"Trainable parameters: "
        f"{stats['trainable']:,}"
    )

    print(
        f"Trainable ratio:      "
        f"{stats['trainable_ratio']:.2%}"
    )

    print()

    for epoch in range(NUM_EPOCHS):

        epoch_start = time.time()

        torch.cuda.reset_peak_memory_stats(
            device
        )

        print(
            f"Epoch {epoch + 1}/{NUM_EPOCHS}"
        )

        train_loss, train_acc = train_one_epoch(
            model,
            train_loader,
            criterion,
            optimizer,
            device,
            scaler
        )

        val_loss, val_acc = evaluate(
            model,
            test_loader,
            criterion,
            device
        )

        epoch_time = (
            time.time() - epoch_start
        )

        peak_memory = (
            torch.cuda.max_memory_allocated(
                device
            ) / 1024**2
        )

        history["train_loss"].append(
            train_loss
        )

        history["train_accuracy"].append(
            train_acc
        )

        history["val_loss"].append(
            val_loss
        )

        history["val_accuracy"].append(
            val_acc
        )

        history["epoch_time"].append(
            epoch_time
        )

        history["gpu_memory_peak_mb"].append(
            peak_memory
        )

        # ----------------------------------------------------
        # Save best checkpoint
        # ----------------------------------------------------

        if val_acc > best_val_accuracy:

            best_val_accuracy = val_acc
            best_epoch = epoch + 1

            torch.save(
                model.state_dict(),
                os.path.join(
                    experiment_dir,
                    "best_model.pt"
                )
            )

        # ----------------------------------------------------
        # Save history after every epoch
        # ----------------------------------------------------

        with open(
            os.path.join(
                experiment_dir,
                "history.json"
            ),
            "w"
        ) as f:

            json.dump(
                history,
                f,
                indent=4
            )

        print(
            f"Train Loss: {train_loss:.4f} | "
            f"Train Acc: {train_acc:.4f}"
        )

        print(
            f"Val Loss:   {val_loss:.4f} | "
            f"Val Acc:   {val_acc:.4f}"
        )

        print(
            f"Time: {epoch_time / 60:.2f} min"
        )

        print(
            f"Peak GPU Memory: "
            f"{peak_memory:.0f} MB"
        )

        print("-" * 70)

    # --------------------------------------------------------
    # Final metadata
    # --------------------------------------------------------

    total_time = (
        time.time() - total_start
    )

    final_results = {
        "best_val_accuracy": best_val_accuracy,
        "best_epoch": best_epoch,
        "final_val_accuracy": history["val_accuracy"][-1],
        "final_train_accuracy": history["train_accuracy"][-1],
        "total_training_time_seconds": total_time,
        "total_training_time_minutes": total_time / 60,
        "average_epoch_time_seconds": np.mean(
            history["epoch_time"]
        ),
        "peak_gpu_memory_mb": max(
            history["gpu_memory_peak_mb"]
        )
    }

    with open(
        os.path.join(
            experiment_dir,
            "results.json"
        ),
        "w"
    ) as f:

        json.dump(
            final_results,
            f,
            indent=4
        )

    print()
    print("=" * 70)
    print("TRAINING COMPLETE")
    print("=" * 70)

    print(
        f"Best validation accuracy: "
        f"{best_val_accuracy:.4f}"
    )

    print(
        f"Best epoch: {best_epoch}"
    )

    print(
        f"Total training time: "
        f"{total_time / 60:.2f} min"
    )


# ============================================================
# Argument parser
# ============================================================

if __name__ == "__main__":

    parser = argparse.ArgumentParser()

    parser.add_argument(
        "--gpu",
        type=int,
        required=True
    )

    parser.add_argument(
        "--pretrained",
        action="store_true"
    )

    parser.add_argument(
        "--train-mode",
        type=str,
        required=True,
        choices=[
            "final_block",
            "full"
        ]
    )

    parser.add_argument(
        "--experiment-name",
        type=str,
        required=True
    )

    args = parser.parse_args()

    train_experiment(
        gpu_id=args.gpu,
        pretrained=args.pretrained,
        train_mode=args.train_mode,
        experiment_name=args.experiment_name
    )

Writing /kaggle/working/train_experiment.py


In [33]:
import os

print(os.path.exists("/kaggle/working/train_experiment.py"))

True
